# 1D Pseudo-Differential Operator Analysis with `psiop`

This notebook demonstrates the capabilities of the `psiop` package for analyzing a 1D pseudo-differential operator. 

We will:
1. Explore the symbol interactively using `interactive_symbol_analysis`.
2. Compute and visualize its pseudospectrum using `pseudospectrum_analysis`.
3. Apply the operator to various test functions (Constant, Linear, Gaussian, Chirp, WKB) to observe its action, support properties, and oscillatory behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# Assuming psiop is in the python path
from psiop import * 

# Define spatial and frequency variables
x, xi = sp.symbols('x xi', real=True)

# Define the symbol 
symbol_expr = 1/((x*xi)**2 + 1)

# Initialize the PseudoDifferentialOperator
op = PseudoDifferentialOperator(
    expr=symbol_expr, 
    vars_x=[x], 
    mode='symbol',
    quantization='kohn-nirenberg'
)

print("Operator symbol:")
sp.pprint(op.symbol)

## 1. Interactive Symbol Analysis

The `interactive_symbol_analysis` function launches an `ipywidgets` dashboard to explore the symbol's amplitude, phase, micro-support, characteristic set, symplectic vector field, and Hamiltonian flow in real-time.

*Note: In a static JSON viewer, the interactive widgets will not render. Please run this notebook in a live Jupyter environment to interact with the sliders.*

In [ ]:
# Launch the interactive dashboard
# We use a spatial domain x in [-5, 5] and frequency xi in [-5, 5]
op.interactive_symbol_analysis(
    xlim=(-5, 5), 
    xi_range=(-5, 5), 
    density=50
)

## 2. Pseudospectrum Analysis

The pseudospectrum $\sigma_\epsilon(P)$ reveals the sensitivity of the spectrum to perturbations, which is particularly rich and highly non-trivial for non-normal operators like the complex Airy operator. 

We compute the resolvent norm $\|(P - \lambda I)^{-1}\|$ over a grid in the complex plane to visualize the $\epsilon$-pseudospectral contours.

In [ ]:
# Define the spatial grid for quantization
# We use a moderately sized grid for the spectral method
x_grid = np.linspace(-10, 10, 128)

# Run the pseudospectrum analysis
# The auto_range feature will adjust the lambda grid based on the computed eigenvalues
pseudo_data = op.pseudospectrum_analysis(
    x_grid=x_grid,
    lambda_real_range=(-2, 2),   # Initial guess, will be auto-adjusted
    lambda_imag_range=(-2, 2),   # Initial guess, will be auto-adjusted
    epsilon_levels=[1e-1, 1e-2, 1e-3, 1e-4],
    resolution=60,               # Grid resolution for lambda
    method='spectral',           # Use spectral method for matrix construction
    auto_range=True,
    plot=True
)

## 3. Application to Test Functions

To understand how the operator acts on different states, we apply $P$ to a set of well-chosen test functions:
1. **Constant function**: $u(x) = 1$ (Tests the multiplicative potential part $ix$)
2. **Linear function**: $u(x) = x$ (Tests both derivative and potential parts)
3. **Gaussian wavepacket**: $u(x) = e^{-x^2/2}$ (Tests localization and smooth decay)
4. **Chirp signal**: $u(x) = e^{i \alpha x^2} e^{-x^2/2}$ (Tests frequency modulation / dispersion)
5. **WKB / Highly oscillatory state**: $u(x) = e^{i k x} e^{-(x-x_0)^2/2}$ (Tests high-frequency asymptotic behavior and transport)

We will plot the real and imaginary parts of the original function and the result $P u(x)$ to observe the support, differential effects, and phase shifts.

In [ ]:
# Setup the grid for application
# We use a large domain and high resolution to avoid periodic wrap-around artifacts
L = 10.0
N = 1024
x_app = np.linspace(-L, L, N, endpoint=False)
dx = x_app[1] - x_app[0]
kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)

# Define test functions
def u_constant(x): return np.ones_like(x)
def u_linear(x): return x
def u_gaussian(x): return np.exp(-0.5 * x**2)
def u_chirp(x): return np.exp(1j * 0.5 * x**2) * np.exp(-0.1 * x**2)
def u_wkb(x): return np.exp(1j * 20.0 * x) * np.exp(-0.5 * (x - 3.0)**2)

test_functions = {
    "Constant": u_constant(x_app),
    "Linear": u_linear(x_app),
    "Gaussian": u_gaussian(x_app),
    "Chirp": u_chirp(x_app),
    "WKB (Highly Oscillatory)": u_wkb(x_app)
}

# Apply the operator and plot
fig, axes = plt.subplots(len(test_functions), 2, figsize=(14, 4 * len(test_functions)))

for i, (name, u) in enumerate(test_functions.items()):
    # Apply operator using the periodic FFT backend.
    # Since the functions decay rapidly or the domain is large, periodic BCs are a good approximation.
    # We use a Gaussian frequency window to prevent high-frequency numerical noise from dominating.
    Pu = op.apply(
        u, 
        x_grid=x_app, 
        kx=kx, 
        boundary_condition='periodic',
        freq_window='gaussian',
        clamp=1e6,

#        backend="direct"
    )
    
    # Plot Original Function
    axes[i, 0].plot(x_app, np.real(u), label='Re(u)', color='blue', lw=2)
    axes[i, 0].plot(x_app, np.imag(u), label='Im(u)', color='red', linestyle='--', lw=2)
    axes[i, 0].set_title(f"Original Function: {name}", fontsize=14)
    axes[i, 0].set_xlabel("x", fontsize=12)
    axes[i, 0].set_ylabel("Amplitude", fontsize=12)
    axes[i, 0].legend(fontsize=10)
    axes[i, 0].grid(True, alpha=0.3)
    
    # Plot Applied Operator (Result)
    axes[i, 1].plot(x_app, np.real(Pu), label='Re(Pu)', color='blue', lw=2)
    axes[i, 1].plot(x_app, np.imag(Pu), label='Im(Pu)', color='red', linestyle='--', lw=2)
    axes[i, 1].set_title(f"Action P(u): {name}", fontsize=14)
    axes[i, 1].set_xlabel("x", fontsize=12)
    axes[i, 1].set_ylabel("Amplitude", fontsize=12)
    axes[i, 1].legend(fontsize=10)
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()